<a href="https://colab.research.google.com/github/UjwalNagrikar/Quant-Finance-Trading-Model-Alorithem-Businees/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
╔══════════════════════════════════════════════════════════════════╗
║   NSE STOCK / INDEX FUTURES  ·  Bollinger Band Backtest          ║
║   AGGRESSIVE VERSION  ·  IST Timestamps Fixed  ·  Full Charges   ║
╚══════════════════════════════════════════════════════════════════╝
"""

# ── stdlib ──────────────────────────────────────────────────────────
import datetime as dt
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# ── third-party ─────────────────────────────────────────────────────
import pytz
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

# ═══════════════════════════════════════════════════════════════════
# 1.  FUTURES DATABASE
# ═══════════════════════════════════════════════════════════════════
DB = {
    # symbol            lot     margin      display name
    "RELIANCE.NS" : (250,  150_000, "Reliance Industries"),
    "TCS.NS"      : (150,  175_000, "Tata Consultancy"),
    "HDFCBANK.NS" : (550,  100_000, "HDFC Bank"),
    "INFY.NS"     : (400,  100_000, "Infosys"),
    "ICICIBANK.NS": (700,   90_000, "ICICI Bank"),
    "SBIN.NS"     : (1500,  80_000, "SBI"),
    "AXISBANK.NS" : (625,   90_000, "Axis Bank"),
    "KOTAKBANK.NS": (400,  130_000, "Kotak Bank"),
    "LT.NS"       : (175,  200_000, "L&T"),
    "TATAMOTORS.NS":(1425,  75_000, "Tata Motors"),
    "TATASTEEL.NS": (5500,  50_000, "Tata Steel"),
    "MARUTI.NS"   : (100,  160_000, "Maruti Suzuki"),
    "WIPRO.NS"    : (1500,  65_000, "Wipro"),
    "BHARTIARTL.NS":(475, 115_000, "Bharti Airtel"),
    "BAJFINANCE.NS":(125,  180_000, "Bajaj Finance"),
    "SUNPHARMA.NS": (700,   95_000, "Sun Pharma"),
    "HCLTECH.NS"  : (700,   85_000, "HCL Tech"),
    "ONGC.NS"     : (3850,  45_000, "ONGC"),
    "NTPC.NS"     : (3000,  42_000, "NTPC"),
    "JSWSTEEL.NS" : (1350,  75_000, "JSW Steel"),
    "M&M.NS"      : (700,   90_000, "M&M"),
    "ADANIENT.NS" : (125,  200_000, "Adani Enterprises"),
    "^NSEI"       : (75,   130_000, "NIFTY 50"),
    "^NSEBANK"    : (15,   125_000, "BANKNIFTY"),
}

# ═══════════════════════════════════════════════════════════════════
# 2.  CONFIG  ← edit here
# ═══════════════════════════════════════════════════════════════════
SYMBOL        = "RELIANCE.NS"
TIMEFRAME     = "1h"
LOOKBACK_DAYS = 365
INITIAL_CAPITAL = 1_000_000     # ₹ 10 Lakh

# Signal mode:  "BREAKOUT" | "BOUNCE" | "BOTH"
SIGNAL_MODE   = "BOTH"

# ── AGGRESSIVE RISK SETTINGS ────────────────────────────────────
RISK_PCT      = 0.05            # 5% capital risked per trade  (was 3%)
STOP_PCT      = 0.012           # 1.2% hard stop               (was 1.5%)
MAX_LOTS      = 10              # max lots per trade            (was 5)
MARGIN_UTIL   = 0.80            # 80% margin utilisation       (was 60%)
COOLDOWN      = 1               # only 1 bar cooldown          (was 2)

# ── TIGHTER BANDS = MORE SIGNALS ────────────────────────────────
BB_LEN        = 14              # shorter period                (was 20)
BB_MULT       = 1.2             # tighter bands = more fires    (was 1.5)

# ── AGGRESSIVE PROFIT TARGET ────────────────────────────────────
USE_TARGET    = True            # enable take-profit
TARGET_PCT    = 0.025           # 2.5% profit target before MR exit

# ── NSE Charges — Zerodha 2024 ──────────────────────────────────
BROKERAGE     = 20.0
STT_RATE      = 0.000125
EXCH_RATE     = 0.000019
SEBI_RATE     = 0.000001
STAMP_RATE    = 0.00002
GST_RATE      = 0.18

# ── Monte Carlo ─────────────────────────────────────────────────
MC_RUNS       = 1_000
MC_RUIN_LVL   = 0.50

# ── Output ──────────────────────────────────────────────────────
OUT_DIR       = "/mnt/user-data/outputs"
IST           = pytz.timezone("Asia/Kolkata")

# ═══════════════════════════════════════════════════════════════════
# 3.  VALIDATE
# ═══════════════════════════════════════════════════════════════════
if SYMBOL not in DB:
    raise ValueError(f"'{SYMBOL}' not in DB.")
LOT_SIZE, MARGIN_LOT, STOCK_NAME = DB[SYMBOL]

# ═══════════════════════════════════════════════════════════════════
# 4.  HELPERS
# ═══════════════════════════════════════════════════════════════════
def charges(price, qty, side):
    tv  = abs(price * qty)
    brk = BROKERAGE
    stt = tv * STT_RATE   if side == "SELL" else 0.0
    exc = tv * EXCH_RATE
    sbi = tv * SEBI_RATE
    stp = tv * STAMP_RATE if side == "BUY"  else 0.0
    gst = (brk + exc + sbi) * GST_RATE
    return brk + stt + exc + sbi + stp + gst

def lot_size(capital, px):
    loss_per_lot   = LOT_SIZE * px * STOP_PCT
    lots_risk      = int(capital * RISK_PCT / loss_per_lot) if loss_per_lot else 0
    lots_margin    = int(capital * MARGIN_UTIL / MARGIN_LOT)
    lots           = min(lots_risk, lots_margin, MAX_LOTS)
    if lots == 0 and lots_margin >= 1:
        lots = 1
    return max(lots, 0)

# ═══════════════════════════════════════════════════════════════════
# 5.  DATA  +  ✅ UTC → IST FIX
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'═'*68}")
print(f"  {STOCK_NAME}  ({SYMBOL})  ·  {TIMEFRAME}  ·  "
      f"BB({BB_LEN},{BB_MULT})  ·  {SIGNAL_MODE}  ·  AGGRESSIVE")
print(f"{'═'*68}")

end   = dt.datetime.now(tz=pytz.utc)
start = end - dt.timedelta(days=LOOKBACK_DAYS + 30)

raw = yf.download(SYMBOL, start=start, end=end,
                  interval=TIMEFRAME, auto_adjust=True, progress=False)
if raw.empty:
    sys.exit("No data returned — check symbol / network.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
raw.rename(columns={"Open":"o","High":"h","Low":"l","Close":"c","Volume":"v"},
           inplace=True)
raw = raw[["o","h","l","c","v"]].copy()

# ── ✅ FIX 1: Convert UTC → IST ─────────────────────────────────
if raw.index.tzinfo is None:
    raw.index = raw.index.tz_localize("UTC")
raw.index = raw.index.tz_convert(IST)

# ── ✅ FIX 2: Keep only NSE market hours 09:15–15:30 IST ────────
raw = raw.between_time("09:15", "15:30")

# ── ✅ FIX 3: Use .loc instead of deprecated .last() ────────────
cutoff = raw.index[-1] - pd.Timedelta(days=LOOKBACK_DAYS)
df = raw.loc[raw.index >= cutoff].copy()

# ── Indicators ──────────────────────────────────────────────────
df["basis"] = df["c"].rolling(BB_LEN).mean()
df["std"]   = df["c"].rolling(BB_LEN).std()
df["upper"] = df["basis"] + BB_MULT * df["std"]
df["lower"] = df["basis"] - BB_MULT * df["std"]
df["ph"]    = df["h"].shift(1)
df["pl"]    = df["l"].shift(1)
df.dropna(inplace=True)

# ── Signals ─────────────────────────────────────────────────────
LB = df["c"] > df["upper"]
SB = df["c"] < df["lower"]
LV = (df["pl"] <= df["lower"]) & (df["c"] > df["lower"])
SV = (df["ph"] >= df["upper"]) & (df["c"] < df["upper"])

if   SIGNAL_MODE == "BREAKOUT": df["ls"] = LB;      df["ss"] = SB
elif SIGNAL_MODE == "BOUNCE":   df["ls"] = LV;      df["ss"] = SV
else:                           df["ls"] = LB | LV; df["ss"] = SB | SV

def slabel(i):
    lb,lv = LB.iloc[i], LV.iloc[i]
    sb,sv = SB.iloc[i], SV.iloc[i]
    if df["ls"].iloc[i]: return "L-BOTH" if lb and lv else ("L-BREAK" if lb else "L-BNCE")
    if df["ss"].iloc[i]: return "S-BOTH" if sb and sv else ("S-BREAK" if sb else "S-BNCE")
    return "-"

nl = int(df["ls"].sum()); ns = int(df["ss"].sum())
print(f"  Bars: {len(df)}   Long sigs: {nl}   Short sigs: {ns}   "
      f"Raw total: {nl+ns}\n")

# ═══════════════════════════════════════════════════════════════════
# 6.  BACKTEST ENGINE
# ═══════════════════════════════════════════════════════════════════
cap   = INITIAL_CAPITAL
side  = None; epx = 0.0; etime = None; esig = ""; nlots = 0; qty = 0.0
cdl   = 0

eq_curve   = []
pnls       = []
trades     = []
total_chg  = 0.0
tno        = 0

for i in range(len(df)):
    row = df.iloc[i]; bt = df.index[i]
    o, h, l, c = row["o"], row["h"], row["l"], row["c"]
    if cdl > 0: cdl -= 1

    # ── EXIT ──────────────────────────────────────────────────────
    if side:
        xpx = None; xrsn = None

        # 1. Hard stop loss
        if side == "LONG"  and l <= epx * (1 - STOP_PCT):
            xpx = round(epx * (1 - STOP_PCT), 2); xrsn = "SL"
        elif side == "SHORT" and h >= epx * (1 + STOP_PCT):
            xpx = round(epx * (1 + STOP_PCT), 2); xrsn = "SL"

        # 2. Aggressive take-profit target
        if xpx is None and USE_TARGET:
            if side == "LONG"  and h >= epx * (1 + TARGET_PCT):
                xpx = round(epx * (1 + TARGET_PCT), 2); xrsn = "TP"
            elif side == "SHORT" and l <= epx * (1 - TARGET_PCT):
                xpx = round(epx * (1 - TARGET_PCT), 2); xrsn = "TP"

        # 3. Mean-reversion to basis
        if xpx is None:
            if (side=="LONG" and c <= row["basis"]) or (side=="SHORT" and c >= row["basis"]):
                xpx = round(o, 2); xrsn = "MR"

        # 4. Opposite signal flip
        if xpx is None:
            if (side=="LONG" and df["ss"].iloc[i]) or (side=="SHORT" and df["ls"].iloc[i]):
                xpx = round(o, 2); xrsn = "FLIP"

        if xpx is not None:
            grs = (xpx - epx) * qty * (1 if side=="LONG" else -1)
            chg = charges(xpx, qty, "SELL" if side=="LONG" else "BUY")
            net = grs - chg
            cap += net; total_chg += chg; pnls.append(net); tno += 1
            try:   bh = i - list(df.index).index(etime)
            except: bh = 0
            trades.append(dict(
                no=tno, side=side, sig=esig,
                edate=str(etime)[:16], xdate=str(bt)[:16], bh=bh,
                epx=epx, xpx=xpx, lots=nlots, qty=int(qty),
                notional=round(epx*qty),
                gross=round(grs,2), chg=round(chg,2), net=round(net,2),
                cap=round(cap), xrsn=xrsn
            ))
            side=None; nlots=0; qty=0.0; etime=None; cdl=COOLDOWN

    # ── ENTRY ─────────────────────────────────────────────────────
    if not side and cdl == 0:
        ns2 = "LONG" if df["ls"].iloc[i] else ("SHORT" if df["ss"].iloc[i] else None)
        if ns2:
            lots = lot_size(cap, o)
            if lots > 0:
                side=ns2; epx=round(o,2); etime=bt; esig=slabel(i)
                nlots=lots; qty=lots*LOT_SIZE
                chg = charges(epx, qty, "BUY" if side=="LONG" else "SELL")
                total_chg += chg; cap -= chg

    # ── EQUITY SNAPSHOT ───────────────────────────────────────────
    unr = (c-epx)*qty*(1 if side=="LONG" else -1) if side else 0.0
    eq_curve.append(cap + unr)

# ═══════════════════════════════════════════════════════════════════
# 7.  METRICS
# ═══════════════════════════════════════════════════════════════════
equity  = pd.Series(eq_curve, index=df.index)
rets    = equity.pct_change().dropna()
dd_ser  = (equity / equity.cummax() - 1) * 100

tot_ret = (equity.iloc[-1] / INITIAL_CAPITAL - 1) * 100
max_dd  = dd_ser.min()
wins    = [p for p in pnls if p > 0]
losses  = [p for p in pnls if p < 0]
wr      = len(wins)/len(pnls)*100 if pnls else 0
aw      = np.mean(wins)   if wins   else 0
al      = np.mean(losses) if losses else 0
pf      = sum(wins)/abs(sum(losses)) if losses else float("inf")
sharpe  = (rets.mean()/rets.std())*np.sqrt(252*6.5) if rets.std()>0 else 0
exp     = (wr/100*aw) + ((1-wr/100)*al)
pct_chg = total_chg / INITIAL_CAPITAL * 100

cw=cl=mcw=mcl=0
for p in pnls:
    if p>0: cw+=1; cl=0; mcw=max(mcw,cw)
    else:   cl+=1; cw=0; mcl=max(mcl,cl)

sig_cnt = {}
for t in trades: sig_cnt[t["sig"]] = sig_cnt.get(t["sig"],0)+1

# ═══════════════════════════════════════════════════════════════════
# 8.  PRINT ALL TRADES
# ═══════════════════════════════════════════════════════════════════
G="\033[92m"; R="\033[91m"; Y="\033[93m"; E="\033[0m"; W="\033[97m"
SEP = "─"*130

print(f"\n{'═'*130}")
print(f"  ALL TRADES  ·  {STOCK_NAME} ({SYMBOL})  ·  {TIMEFRAME}  ·  "
      f"BB({BB_LEN},{BB_MULT})  ·  {SIGNAL_MODE}  ·  SL {STOP_PCT*100:.1f}%  ·  TP {TARGET_PCT*100:.1f}%  ·  IST")
print(f"{'═'*130}")
print(f"  {'#':>4}  {'':4}  {'Side':<6}  {'Sig':<8}  {'Entry (IST)':^16}  {'Exit (IST)':^16}  "
      f"{'Bars':>4}  {'Entry₹':>9}  {'Exit₹':>9}  {'Lots':>4}  "
      f"{'Notional₹':>12}  {'Gross₹':>10}  {'Chg₹':>7}  {'Net₹':>10}  "
      f"{'Capital₹':>12}  Rsn")
print(SEP)

for t in trades:
    ok  = t["net"] >= 0
    nc  = G if ok else R
    lbl = f"{nc}{'WIN ':>4}{E}" if ok else f"{nc}{'LOSS':>4}{E}"
    print(
        f"  {t['no']:>4}  {lbl}  {t['side']:<6}  {t['sig']:<8}  "
        f"{t['edate']:^16}  {t['xdate']:^16}  {t['bh']:>4}  "
        f"₹{t['epx']:>8,.1f}  ₹{t['xpx']:>8,.1f}  {t['lots']:>4}  "
        f"₹{t['notional']:>11,.0f}  {t['gross']:>+10,.0f}  {t['chg']:>7,.0f}  "
        f"{nc}{t['net']:>+10,.0f}{E}  "
        f"₹{t['cap']:>11,.0f}  {t['xrsn']}"
    )

g_tot = sum(t["gross"] for t in trades)
n_tot = sum(t["net"]   for t in trades)
nc = G if n_tot >= 0 else R
print(SEP)
print(f"  {'TOT':>4}  {'':4}  {'':6}  {'':8}  {'':16}  {'':16}  {'':4}  "
      f"  {'':9}  {'':9}  {'':4}  "
      f"  {'':12}  {g_tot:>+10,.0f}  {total_chg:>7,.0f}  "
      f"{nc}{n_tot:>+10,.0f}{E}  ₹{equity.iloc[-1]:>11,.0f}")
print(f"{'═'*130}")

# ═══════════════════════════════════════════════════════════════════
# 9.  PERFORMANCE SUMMARY
# ═══════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print(f"  PERFORMANCE SUMMARY  ·  {STOCK_NAME}  [AGGRESSIVE]")
print(f"{'═'*60}")
def row(lbl, val, col=""):
    print(f"  {lbl:<28}  {col}{val}{E}")

row("Symbol · Timeframe",  f"{SYMBOL}  ·  {TIMEFRAME}")
row("Period (IST)",        f"{df.index[0].strftime('%d %b %Y %H:%M')} → {df.index[-1].strftime('%d %b %Y %H:%M')}")
row("Signal Mode",         SIGNAL_MODE)
row("BB",                  f"({BB_LEN}, {BB_MULT}σ)  Lot={LOT_SIZE}  Margin=₹{MARGIN_LOT:,.0f}")
row("Risk · Stop · Target",f"{RISK_PCT*100:.1f}%  ·  {STOP_PCT*100:.1f}%  ·  {TARGET_PCT*100:.1f}%  Cooldown={COOLDOWN}bar")
print(f"  {'─'*56}")
row("Initial Capital",     f"₹{INITIAL_CAPITAL:>14,.0f}")
fc = G if tot_ret>=0 else R
row("Final Capital",       f"₹{equity.iloc[-1]:>14,.0f}", fc)
row("Total Return",        f"{tot_ret:>+14.2f}%", fc)
row("Net P&L",             f"₹{equity.iloc[-1]-INITIAL_CAPITAL:>+14,.0f}", fc)
print(f"  {'─'*56}")
row("Total Trades",        f"{len(pnls):>14}")
row("Wins / Losses",       f"{len(wins):>6}  /  {len(losses):<6}   WR {wr:.1f}%")
row("Avg Win",             f"₹{aw:>+14,.0f}", G)
row("Avg Loss",            f"₹{al:>+14,.0f}", R)
row("Expectancy / trade",  f"₹{exp:>+14,.0f}", G if exp>0 else R)
row("Profit Factor",       f"{pf:>14.2f}", G if pf>1.5 else Y if pf>1 else R)
row("Best / Worst trade",  f"₹{max(pnls) if pnls else 0:>+12,.0f}  /  ₹{min(pnls) if pnls else 0:>+12,.0f}")
row("Max Consec W / L",    f"{mcw:>6}  /  {mcl}")
print(f"  {'─'*56}")
row("Sharpe (ann.)",       f"{sharpe:>14.2f}", G if sharpe>1 else Y if sharpe>0 else R)
row("Max Drawdown",        f"{max_dd:>13.2f}%", R if max_dd<-20 else Y)
print(f"  {'─'*56}")
print("  Signal breakdown:")
for sig,cnt in sorted(sig_cnt.items(), key=lambda x:-x[1]):
    ww = sum(1 for t in trades if t["sig"]==sig and t["net"]>0)
    print(f"    {sig:<12}  {cnt:>4} trades   WR {ww/cnt*100:.0f}%")
print(f"  {'─'*56}")
row("Total Charges",       f"₹{total_chg:>14,.0f}  ({pct_chg:.2f}% of cap)", Y)
print(f"{'═'*60}\n")

# Running equity per trade
print(f"{'═'*70}")
print("  RUNNING EQUITY  (after each closed trade)  — IST")
print(f"{'═'*70}")
print(f"  {'#':>4}  {'Date (IST)':<16}  {'Side':<6}  {'Net₹':>10}  "
      f"{'Capital₹':>12}  {'vs Start':>9}  {'DD from Peak':>13}  Rsn")
print(f"  {'─'*66}")
peak = INITIAL_CAPITAL
for t in trades:
    peak = max(peak, t["cap"])
    ddfp = (t["cap"]/peak - 1)*100
    vs   = (t["cap"]/INITIAL_CAPITAL - 1)*100
    nc   = G if t["net"]>=0 else R
    dc   = R if ddfp<-5 else Y if ddfp<0 else G
    print(f"  {t['no']:>4}  {t['xdate'][:16]:<16}  {t['side']:<6}  "
          f"{nc}{t['net']:>+10,.0f}{E}  ₹{t['cap']:>11,.0f}  "
          f"{nc}{vs:>+8.2f}%{E}  {dc}{ddfp:>12.2f}%{E}  {t['xrsn']}")
print(f"{'═'*70}\n")

# ═══════════════════════════════════════════════════════════════════
# 10.  CHART 1 — EQUITY DASHBOARD (6-panel)
# ═══════════════════════════════════════════════════════════════════
os.makedirs(OUT_DIR, exist_ok=True)
sym = SYMBOL.replace(".NS","").replace("^","")

BG   = "#07090f"; CARD = "#0d1120"; GRID = "#161d30"
TXT  = "#dde3f0"; DIM  = "#4a5570"
BLU  = "#4f8ef7"; GRN  = "#22c55e"; RED  = "#ef4444"
YLW  = "#f59e0b"; WHT  = "#f0f4ff"

fig = plt.figure(figsize=(24, 16), facecolor=BG)
gs  = gridspec.GridSpec(
    4, 3, figure=fig,
    height_ratios=[2.6, 1.0, 0.75, 0.85],
    width_ratios=[2.0, 1.1, 1.1],
    hspace=0.07, wspace=0.20,
    left=0.055, right=0.975, top=0.905, bottom=0.06
)

ax_eq  = fig.add_subplot(gs[0, :2])
ax_pnl = fig.add_subplot(gs[0, 2])
ax_dd  = fig.add_subplot(gs[1, :2], sharex=ax_eq)
ax_pie = fig.add_subplot(gs[1, 2])
ax_mon = fig.add_subplot(gs[2, :2], sharex=ax_eq)
ax_kpi = fig.add_subplot(gs[3, :])

for ax in (ax_eq, ax_pnl, ax_dd, ax_pie, ax_mon, ax_kpi):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=DIM, labelsize=8)
    for sp in ax.spines.values(): sp.set_color(GRID)

ev = equity.values; ei = equity.index

# A. Equity curve
ax_eq.plot(ei, ev, lw=2.0, color=BLU, zorder=4)
ax_eq.fill_between(ei, INITIAL_CAPITAL, ev,
                   where=ev >= INITIAL_CAPITAL, interpolate=True,
                   color=GRN, alpha=0.20, zorder=2)
ax_eq.fill_between(ei, INITIAL_CAPITAL, ev,
                   where=ev < INITIAL_CAPITAL, interpolate=True,
                   color=RED, alpha=0.25, zorder=2)
ax_eq.axhline(INITIAL_CAPITAL, color=WHT, lw=0.7, ls=(0,(5,4)), alpha=0.30)
ax_eq.plot(ei, equity.cummax().values, lw=0.7, color=WHT, alpha=0.12, ls="--")

for t in trades:
    mask = equity.index.astype(str).str.startswith(t["xdate"][:10])
    if mask.any():
        ax_eq.scatter(equity.index[mask][0], equity[mask].iloc[0],
                      color=GRN if t["net"]>=0 else RED,
                      s=22, zorder=6, alpha=0.75, linewidths=0)

fv = equity.iloc[-1]; fc2 = GRN if fv >= INITIAL_CAPITAL else RED
ax_eq.annotate(
    f"  Rs{fv:,.0f}",
    xy=(ei[-1], fv), xytext=(-85, 20), textcoords="offset points",
    fontsize=11, color=fc2, fontweight="bold",
    arrowprops=dict(arrowstyle="-", color=fc2, lw=1.0, alpha=0.6)
)
rc = GRN if tot_ret >= 0 else RED
ax_eq.text(0.99, 0.96, f"{tot_ret:+.2f}%", transform=ax_eq.transAxes,
           fontsize=17, fontweight="bold", color=rc, ha="right", va="top",
           bbox=dict(boxstyle="round,pad=0.35", fc=BG, ec=rc, lw=1.4, alpha=0.88))

ax_eq.set_ylabel("Portfolio Value  (INR)", color=TXT, fontsize=9, labelpad=6)
ax_eq.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"Rs{x/1e5:.1f}L"))
ax_eq.grid(color=GRID, lw=0.45); ax_eq.tick_params(labelbottom=False)
ax_eq.set_xlim(ei[0], ei[-1])

# B. Drawdown
dv = dd_ser.values
ax_dd.fill_between(dd_ser.index, 0, dv, color=RED, alpha=0.50)
ax_dd.plot(dd_ser.index, dv, lw=0.85, color="#ff7070")
ax_dd.axhline(0, color=GRID, lw=0.8)
mi = dd_ser.idxmin()
ax_dd.annotate(f"{max_dd:.1f}%", xy=(mi, dd_ser.min()),
               xytext=(18,-13), textcoords="offset points",
               fontsize=8, color=RED, fontweight="bold",
               arrowprops=dict(arrowstyle="->", color=RED, lw=0.9))
ax_dd.set_ylabel("Drawdown", color=TXT, fontsize=8, labelpad=6)
ax_dd.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}%"))
ax_dd.set_ylim(dv.min()*1.35, 0.5)
ax_dd.grid(color=GRID, lw=0.4); ax_dd.tick_params(labelbottom=False)

# C. Monthly returns
meq = equity.resample("ME").last()
mr  = meq.pct_change().dropna() * 100
ax_mon.bar(mr.index, mr.values,
           color=[GRN if v>=0 else RED for v in mr.values],
           alpha=0.80, width=20)
ax_mon.axhline(0, color=GRID, lw=0.8)
ax_mon.set_ylabel("Monthly %", color=TXT, fontsize=7.5, labelpad=6)
ax_mon.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:+.0f}%"))
ax_mon.grid(color=GRID, lw=0.3)
ax_mon.set_xlabel("Date (IST)", color=DIM, fontsize=8)

# D. Per-trade P&L bars
if trades:
    tnos  = [t["no"]  for t in trades]
    tnets = [t["net"] for t in trades]
    ax_pnl.bar(tnos, tnets,
               color=[GRN if n>=0 else RED for n in tnets],
               alpha=0.80, width=0.8)
    ax_pnl.axhline(0, color=GRID, lw=0.8)
    ax_r = ax_pnl.twinx()
    ax_r.set_facecolor(CARD)
    ax_r.plot(tnos, pd.Series(tnets).expanding().mean(),
              color=BLU, lw=1.4, ls="--", alpha=0.8)
    ax_r.tick_params(colors=DIM, labelsize=7)
    ax_r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"Rs{x/1e3:.0f}K"))
    ax_r.set_ylabel("Cum avg", color=BLU, fontsize=7)
    for sp in ax_r.spines.values(): sp.set_color(GRID)
ax_pnl.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"Rs{x/1e3:.0f}K"))
ax_pnl.set_xlabel("Trade #", color=DIM, fontsize=8)
ax_pnl.set_ylabel("Net P&L  (INR)", color=TXT, fontsize=8)
ax_pnl.set_title("Per-Trade P&L", color=TXT, fontsize=9, fontweight="bold", pad=5)
ax_pnl.grid(color=GRID, lw=0.35)

# E. Win/loss donut
nw = len(wins); nl2 = len(losses)
if nw + nl2 > 0:
    ax_pie.pie([nw, nl2], colors=[GRN, RED], startangle=90,
               wedgeprops=dict(width=0.52, edgecolor=CARD, linewidth=2.5))
    ax_pie.text(0, 0.08, f"{wr:.1f}%",  ha="center", fontsize=13,
                color=TXT, fontweight="bold")
    ax_pie.text(0,-0.20, "Win Rate", ha="center", fontsize=7.5, color=DIM)
    ax_pie.text(-0.85, 0.65, f"W:{nw}", fontsize=8, color=GRN)
    ax_pie.text( 0.45, 0.65, f"L:{nl2}", fontsize=8, color=RED)
ax_pie.set_title("Win / Loss", color=TXT, fontsize=9, fontweight="bold", pad=5)

# F. KPI strip
ax_kpi.axis("off")
kpis = [
    ("Initial Capital", f"Rs{INITIAL_CAPITAL/1e5:.0f}L",     WHT),
    ("Final Capital",   f"Rs{equity.iloc[-1]/1e5:.2f}L",     fc2),
    ("Total Return",    f"{tot_ret:+.2f}%",                   fc2),
    ("Net P&L",         f"Rs{(equity.iloc[-1]-INITIAL_CAPITAL)/1e3:+.1f}K", fc2),
    ("Trades",          f"{len(pnls)}",                       TXT),
    ("Win Rate",        f"{wr:.1f}%",                         GRN if wr>55 else YLW if wr>45 else RED),
    ("Profit Factor",   f"{pf:.2f}",                          GRN if pf>1.5 else YLW if pf>1 else RED),
    ("Sharpe",          f"{sharpe:.2f}",                      GRN if sharpe>1 else YLW if sharpe>0 else RED),
    ("Max Drawdown",    f"{max_dd:.2f}%",                     RED if max_dd<-20 else YLW),
    ("Expectancy",      f"Rs{exp/1e3:+.1f}K/tr",             GRN if exp>0 else RED),
    ("Charges",         f"Rs{total_chg/1e3:.1f}K ({pct_chg:.1f}%)", YLW),
]
n = len(kpis); sp = 1.0/n
for k,(lbl,val,col) in enumerate(kpis):
    cx = k*sp + sp*0.04; cw = sp*0.92
    rect = plt.Rectangle((cx,0.06), cw, 0.88, facecolor=GRID,
                          edgecolor=col, lw=0.9, alpha=0.65,
                          transform=ax_kpi.transAxes, clip_on=False)
    ax_kpi.add_patch(rect)
    ax_kpi.text(cx+cw/2, 0.70, lbl, transform=ax_kpi.transAxes,
                ha="center", fontsize=6.8, color=DIM)
    ax_kpi.text(cx+cw/2, 0.28, val, transform=ax_kpi.transAxes,
                ha="center", fontsize=9.5, color=col, fontweight="bold")

period = f"{df.index[0].strftime('%d %b %Y %H:%M')}  to  {df.index[-1].strftime('%d %b %Y %H:%M')}  IST"
fig.text(0.055, 0.945, f"{STOCK_NAME}  ({SYMBOL})  [AGGRESSIVE]",
         fontsize=16, fontweight="bold", color=WHT)
fig.text(0.055, 0.924, f"Bollinger Band Futures Backtest  ·  {TIMEFRAME}  ·  "
         f"BB({BB_LEN},{BB_MULT})  ·  {SIGNAL_MODE}  ·  {period}",
         fontsize=9, color=DIM)
fig.text(0.975, 0.012,
         "Backtest results are hypothetical. Not financial advice.",
         fontsize=7, color=DIM, ha="right", style="italic")

out1 = f"{OUT_DIR}/equity_{sym}_{TIMEFRAME}_aggressive.png"
fig.savefig(out1, dpi=155, bbox_inches="tight", facecolor=BG)
print(f"[OK]  Equity chart  ->  {out1}")
plt.close(fig)

# ═══════════════════════════════════════════════════════════════════
# 11.  CHART 2 — MONTE CARLO
# ═══════════════════════════════════════════════════════════════════
if len(pnls) < 5:
    print("Too few trades for Monte Carlo (need >= 5).")
else:
    rng2    = np.random.default_rng(42)
    parr    = np.array(pnls)
    ntr     = len(parr)
    paths   = np.empty((MC_RUNS, ntr+1))
    paths[:,0] = INITIAL_CAPITAL
    fc_arr  = np.empty(MC_RUNS)
    dd_arr  = np.empty(MC_RUNS)
    ruin    = np.zeros(MC_RUNS, bool)

    print(f"\n  Running {MC_RUNS:,} Monte Carlo paths", end="", flush=True)
    for s in range(MC_RUNS):
        shuf      = rng2.choice(parr, size=ntr, replace=True)
        path      = np.concatenate([[INITIAL_CAPITAL], INITIAL_CAPITAL + np.cumsum(shuf)])
        paths[s]  = path
        fc_arr[s] = path[-1]
        pk        = np.maximum.accumulate(path)
        dd_arr[s] = ((path/pk - 1)).min() * 100
        if np.any(path < INITIAL_CAPITAL * MC_RUIN_LVL):
            ruin[s] = True
        if (s+1) % 200 == 0: print(".", end="", flush=True)
    print(" done!")

    p5  = np.percentile(paths,  5, axis=0)
    p25 = np.percentile(paths, 25, axis=0)
    p50 = np.percentile(paths, 50, axis=0)
    p75 = np.percentile(paths, 75, axis=0)
    p95 = np.percentile(paths, 95, axis=0)
    xa  = np.arange(ntr+1)

    ruin_pct  = ruin.mean()*100
    prof_pct  = (fc_arr > INITIAL_CAPITAL).mean()*100
    act_beats = (fc_arr < equity.iloc[-1]).mean()*100
    med_fc    = np.median(fc_arr)
    p5fc      = np.percentile(fc_arr,  5)
    p95fc     = np.percentile(fc_arr, 95)
    med_dd    = np.median(dd_arr)
    p95dd     = np.percentile(dd_arr, 95)

    actual_path = np.array([INITIAL_CAPITAL]+list(INITIAL_CAPITAL+np.cumsum(pnls)))

    fig2 = plt.figure(figsize=(22, 12), facecolor=BG)
    gs2  = gridspec.GridSpec(2, 2, figure=fig2,
                             hspace=0.28, wspace=0.22,
                             left=0.07, right=0.97, top=0.90, bottom=0.07)
    axs  = [fig2.add_subplot(gs2[r,c]) for r in range(2) for c in range(2)]
    for ax in axs:
        ax.set_facecolor(CARD)
        ax.tick_params(colors=DIM, labelsize=8)
        for sp in ax.spines.values(): sp.set_color(GRID)

    # Panel 1 – fan chart
    ax1 = axs[0]
    sidx = rng2.choice(MC_RUNS, size=min(300,MC_RUNS), replace=False)
    for s in sidx:
        clr = "#1d4429" if paths[s,-1]>=INITIAL_CAPITAL else "#3d1515"
        ax1.plot(xa, paths[s], lw=0.25, color=clr, alpha=0.4, zorder=1)
    ax1.fill_between(xa, p5,  p95, alpha=0.18, color=BLU, label="P5-P95")
    ax1.fill_between(xa, p25, p75, alpha=0.32, color=BLU, label="P25-P75")
    ax1.plot(xa, p50, lw=2.0, color=YLW, label="Median", zorder=4)
    ax1.plot(xa, p95, lw=1.1, color=GRN, ls="--", label="P95", zorder=4)
    ax1.plot(xa, p5,  lw=1.1, color=RED, ls="--", label="P5",  zorder=4)
    ax1.plot(xa, actual_path, lw=2.2, color=WHT,
             label=f"Actual  {tot_ret:+.1f}%", zorder=5)
    ax1.axhline(INITIAL_CAPITAL,             color=DIM, lw=0.8, ls=":")
    ax1.axhline(INITIAL_CAPITAL*MC_RUIN_LVL, color=RED, lw=0.9, ls=":", alpha=0.55,
                label="Ruin level")
    ax1.set_title("Equity Fan Chart  (1 000 paths)", color=TXT, fontsize=10, fontweight="bold")
    ax1.set_xlabel("Trade #", color=DIM, fontsize=8)
    ax1.set_ylabel("Capital  (INR)", color=TXT, fontsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"Rs{x/1e5:.1f}L"))
    ax1.legend(fontsize=7, facecolor=BG, edgecolor=GRID, labelcolor=DIM, ncol=2)
    ax1.grid(color=GRID, lw=0.4)

    # Panel 2 – final capital histogram
    ax2 = axs[1]
    bins = np.linspace(fc_arr.min(), fc_arr.max(), 65)
    wm   = fc_arr >= INITIAL_CAPITAL
    ax2.hist(fc_arr[wm],  bins=bins, color=GRN, alpha=0.72, label="Profitable")
    ax2.hist(fc_arr[~wm], bins=bins, color=RED, alpha=0.72, label="Loss")
    for val, lbl, clr in [
        (p5fc,  "P5",     RED),
        (med_fc,"Median", YLW),
        (p95fc, "P95",    GRN),
        (equity.iloc[-1], "Actual", WHT),
        (INITIAL_CAPITAL, "Start",  DIM),
    ]:
        ax2.axvline(val, color=clr, lw=1.4, ls="--")
        ax2.text(val, ax2.get_ylim()[1]*0.02,
                 f" {lbl}\n Rs{val/1e5:.1f}L", color=clr, fontsize=6.5)
    ax2.set_title("Final Capital Distribution", color=TXT, fontsize=10, fontweight="bold")
    ax2.set_xlabel("Final Capital", color=DIM, fontsize=8)
    ax2.set_ylabel("Count", color=TXT, fontsize=8)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"Rs{x/1e5:.0f}L"))
    ax2.legend(fontsize=7.5, facecolor=BG, edgecolor=GRID, labelcolor=DIM)
    ax2.grid(color=GRID, lw=0.4)

    # Panel 3 – max drawdown histogram
    ax3 = axs[2]
    ddbins = np.linspace(dd_arr.min(), 0, 55)
    ax3.hist(dd_arr, bins=ddbins, color=RED, alpha=0.72, edgecolor=BG, lw=0.3)
    ax3.axvline(max_dd,      color=WHT, lw=1.8, ls="-",  label=f"Actual {max_dd:.1f}%")
    ax3.axvline(med_dd,      color=YLW, lw=1.4, ls="--", label=f"Median {med_dd:.1f}%")
    ax3.axvline(p95dd,       color=YLW, lw=1.1, ls=":",  label=f"P95 {p95dd:.1f}%")
    ax3.set_title("Max Drawdown Distribution", color=TXT, fontsize=10, fontweight="bold")
    ax3.set_xlabel("Max Drawdown (%)", color=DIM, fontsize=8)
    ax3.set_ylabel("Count", color=TXT, fontsize=8)
    ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}%"))
    ax3.legend(fontsize=7.5, facecolor=BG, edgecolor=GRID, labelcolor=DIM)
    ax3.grid(color=GRID, lw=0.4)

    # Panel 4 – scorecard
    ax4 = axs[3]; ax4.axis("off")
    rc2 = "lime" if ruin_pct<5 else ("orange" if ruin_pct<20 else "red")
    pc2 = "lime" if prof_pct>70 else ("orange" if prof_pct>50 else "red")
    lines = [
        ("MONTE CARLO  SCORECARD", WHT,  13, "bold"),
        ("", DIM, 8, "normal"),
        (f"Simulations       {MC_RUNS:,}", DIM, 9, "normal"),
        (f"Trades / path     {ntr}",       DIM, 9, "normal"),
        (f"Method            bootstrap resample", DIM, 9, "normal"),
        ("-- Final Capital -------------------------", DIM, 8, "normal"),
        (f"P95   Rs{p95fc/1e5:.2f}L  ({(p95fc/INITIAL_CAPITAL-1)*100:+.1f}%)", GRN,  9, "normal"),
        (f"P50   Rs{med_fc/1e5:.2f}L  ({(med_fc/INITIAL_CAPITAL-1)*100:+.1f}%)", YLW, 9, "bold"),
        (f"P5    Rs{p5fc/1e5:.2f}L  ({(p5fc/INITIAL_CAPITAL-1)*100:+.1f}%)",  RED,  9, "normal"),
        (f"Actual  Rs{equity.iloc[-1]/1e5:.2f}L  -> beats {act_beats:.0f}% paths", WHT, 9, "bold"),
        ("-- Risk ----------------------------------", DIM, 8, "normal"),
        (f"Profitable paths      {prof_pct:.1f}%",  pc2, 10, "bold"),
        (f"Ruin probability      {ruin_pct:.1f}%",  rc2, 10, "bold"),
        (f"Median max DD         {med_dd:.2f}%",    DIM,  9, "normal"),
        (f"P95 max DD            {p95dd:.2f}%",     RED,  9, "normal"),
        ("-- Verdict -------------------------------", DIM, 8, "normal"),
    ]
    if ruin_pct < 5:   lines.append(("Low ruin — robust strategy",     "lime",   9, "normal"))
    elif ruin_pct<20:  lines.append(("Moderate ruin — tighten risk",    "orange", 9, "normal"))
    else:              lines.append(("HIGH ruin — reduce size!",         "red",    9, "bold"))
    if prof_pct > 70:  lines.append(("Real statistical edge",           "lime",   9, "normal"))
    elif prof_pct>50:  lines.append(("Marginal edge — need more data",  "orange", 9, "normal"))
    else:              lines.append(("No clear edge detected",            "red",    9, "bold"))

    y = 0.97
    for txt, col, sz, wt in lines:
        ax4.text(0.04, y, txt, transform=ax4.transAxes,
                 fontsize=sz, color=col, fontweight=wt, va="top",
                 fontfamily="monospace")
        y -= 0.048

    fig2.suptitle(
        f"Monte Carlo Simulation  ·  {STOCK_NAME} ({SYMBOL})  [AGGRESSIVE]  ·  "
        f"{MC_RUNS:,} paths  ·  {ntr} trades/path",
        color=WHT, fontsize=13, fontweight="bold"
    )
    out2 = f"{OUT_DIR}/montecarlo_{sym}_{TIMEFRAME}_aggressive.png"
    fig2.savefig(out2, dpi=155, bbox_inches="tight", facecolor=BG)
    print(f"[OK]  Monte Carlo chart  ->  {out2}")
    plt.close(fig2)

print(f"\n[OK]  All done!  {len(pnls)} trades on {STOCK_NAME}\n")


════════════════════════════════════════════════════════════════════
  Reliance Industries  (RELIANCE.NS)  ·  1h  ·  BB(14,1.2)  ·  BOTH  ·  AGGRESSIVE
════════════════════════════════════════════════════════════════════
  Bars: 1705   Long sigs: 626   Short sigs: 608   Raw total: 1234


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
  ALL TRADES  ·  Reliance Industries (RELIANCE.NS)  ·  1h  ·  BB(14,1.2)  ·  BOTH  ·  SL 1.2%  ·  TP 2.5%  ·  IST
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
     #        Side    Sig         Entry (IST)        Exit (IST)     Bars     Entry₹      Exit₹  Lots     Notional₹      Gross₹     Chg₹        Net₹      Capital₹  Rsn
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     1  WIN   LONG    L-BREAK   2025

Test ans Optimize
